# Deteção de Anomalias em Dados Cripto Multi-Exchange
## Processamento de Big Data — Tema 5 (Reescrita Big Data)

| Campo | Detalhe |
|---|---|
| **Docente** | Pedro Sobreiro |
| **UC** | Processamento de Big Data |
| **Ano Letivo** | 2025/2026 |
| **Tema** | Tema 5 — Anomalias Multi-Exchange |
| **Dados** | BTC/USDT real — Binance, Bybit, Coinbase (ccxt) |

---

**Referências (APA 7.ª edição)**

Apache Software Foundation. (2024). *PySpark API reference (v. 3.5)*. https://spark.apache.org/docs/latest/api/python/

Zaharia, M., Xin, R. S., Wendell, P., Das, T., Armbrust, M., Dave, A., Meng, X., Rosen, J., Venkataraman, S., Franklin, M. J., Ghodsi, A., Gonzalez, J., Shenker, S., & Stoica, I. (2016). Apache Spark: A unified engine for big data processing. *Communications of the ACM, 59*(11), 56–65. https://doi.org/10.1145/2934664

Zhang, T., Ramakrishnan, R., & Livny, M. (1996). BIRCH: An efficient data clustering method for very large databases. *ACM SIGMOD Record, 25*(2), 103–114.

Sobreiro, P. (2025). *big_data* [Repositório GitHub]. https://github.com/pesobreiro/big_data


---
## 1. Introdução

O mercado de criptomoedas opera ininterruptamente em dezenas de exchanges ao mesmo tempo.
Esta fragmentação cria oportunidades para fenómenos anómalos — *flash crashes*, picos de volume,
e divergências de preço entre plataformas (*cross-exchange spread* incomum) — que constituem
sinais de arbitragem, manipulação ou falha técnica.

**Dataset:** OHLCV real de BTC/USDT extraído com a biblioteca `ccxt`, armazenado em três ficheiros
Parquet locais: `data/binance_btc.parquet`, `data/bybit_btc.parquet`, `data/coinbase_btc.parquet`.
Cada ficheiro contém mais de 100 000 linhas com o schema:
`timestamp` (datetime64[us]), `open`, `high`, `low`, `close`, `volume`.

**Hipótese:** Um pipeline combinando Z-score distribuído (Spark SQL/Window) com
BisectingKMeans nativo do PySpark ML é capaz de detetar anomalias de forma
completamente distribuída, sem qualquer invocação de `.toPandas()` no caminho de scoring.

**Arquitetura (Medallion):**
```
Bronze  →  ficheiros Parquet brutos por exchange (produzidos externamente por ccxt)
Silver  →  alinhamento temporal + feature engineering (Parquet)
Gold    →  scores de anomalia + flags combinados (Parquet particionado)
```

**Alterações arquitecturais face ao template original:**

| Elemento original | Substituição |
|---|---|
| Geração sintética GBM | Leitura de Parquet reais (ccxt) |
| `sklearn.IsolationForest` via `.toPandas()` | `pyspark.ml.clustering.BisectingKMeans` (nativo) |
| `sklearn.StandardScaler` | `pyspark.ml.feature.StandardScaler` (distribuído) |
| Back-join Pandas → Spark | Eliminado — todo o scoring é Spark |


---
## 2. Metodologia

### 2.1 Setup — Imports, SparkSession e Configuração


In [87]:
# ── Instalar PySpark no Colab (remover comentário se necessário) ──────────────
# !pip install pyspark

import os

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType

from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import BisectingKMeans
from pyspark.ml import Pipeline

# ── Directórios ───────────────────────────────────────────────────────────────
BASE   = os.getcwd()
SILVER = os.path.join(BASE, "data", "silver")
GOLD   = os.path.join(BASE, "data", "gold")
for d in [SILVER, GOLD]:
    os.makedirs(d, exist_ok=True)

# ── Paths dos ficheiros Bronze (produzidos externamente pelo ccxt) ─────────────
# Schema esperado por ficheiro:
#   timestamp  datetime64[us]
#   open, high, low, close, volume  float64
BRONZE_FILES = {
    "binance":  os.path.join(BASE, "data", "binance_btc.parquet"),
    "kucoin":    os.path.join(BASE, "data", "kucoin_btc.parquet"),
    "coinbase": os.path.join(BASE, "data", "coinbase_btc.parquet"),
}

# ── Hiper-parâmetros do pipeline ──────────────────────────────────────────────

# Features usadas no detector Z-score
Z_FEATURES = [
    "returns_binance",
    "volatility7_binance",
    "vol_ratio_binance",
    "max_cross_gap",
    "price_range_binance",
]
Z_THRESH = 3.5   # limiar |z| para flag de anomalia

# Features completas para o pipeline ML (espelha IF_FEATURES do notebook original)
ML_FEATURES = [
    "returns_binance",    "volatility7_binance",
    "vol_ratio_binance",  "max_cross_gap",        "price_range_binance",
    "returns_kucoin",      "vol_ratio_kucoin",
    "returns_coinbase",   "vol_ratio_coinbase",
]

# BisectingKMeans
BKM_K                = 8     # número de clusters folha
BKM_MAX_ITER         = 20    # iterações máximas de bissecção
BKM_SEED             = 42
ANOMALY_CLUSTER_FRAC = 0.02  # clusters com ≤ 2% da população → anómalos
                              # (espelha contamination=0.02 do IsolationForest original)

# ── SparkSession ──────────────────────────────────────────────────────────────
# Para cluster real: remover .master("local[*]") e submeter via spark-submit.
# shuffle.partitions deve ser 2–3× o número de cores do cluster.
spark = (
    SparkSession.builder
    .appName("Tema5_AnomaliasCrypto_BigData")
    .config("spark.sql.ansi.enabled", "false")
    .config("spark.driver.memory", "4g")
    # ~128 MB por partição após shuffle — ajustar conforme o cluster
    .config("spark.sql.shuffle.partitions", "32")
    # Adaptive Query Execution (Spark 3.x): coalesce automático de partições pequenas
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"PySpark {spark.version} — sessão pronta")
print(f"UI: {spark.sparkContext.uiWebUrl}")


PySpark 3.5.0 — sessão pronta
UI: http://caa1bbe77a5f:4040


---
### 2.2 Camada Silver — Ingestão e Alinhamento Temporal

Leitura dos três ficheiros Parquet Bronze e *join* interno por `timestamp`.
Apenas os instantes presentes nas três exchanges simultaneamente são retidos —
condição necessária para que as features de divergência inter-exchange sejam
numericamente válidas.


In [88]:
def read_and_rename(exchange: str, path: str):
    """
    Lê um ficheiro Parquet Bronze, garante o tipo correto da coluna timestamp,
    e renomeia as colunas OHLCV com o sufixo do exchange.

    Parâmetros
    ----------
    exchange : str   Sufixo de coluna (e.g. "binance").
    path     : str   Caminho absoluto para o ficheiro Parquet.

    Retorna
    -------
    pyspark.sql.DataFrame
        Colunas: timestamp | open_<exch> | high_<exch> | low_<exch>
                 | close_<exch> | volume_<exch>
    """
    sdf = spark.read.parquet(path)

    # Cast defensivo: ccxt pode escrever INT96 ou TIMESTAMP_MICROS no Parquet
    sdf = sdf.withColumn("timestamp", F.col("timestamp").cast("timestamp"))

    # Renomear OHLCV com sufixo do exchange para evitar colisões no join
    for col in ["open", "high", "low", "close", "volume"]:
        sdf = sdf.withColumnRenamed(col, f"{col}_{exchange}")

    # Estatísticas descritivas rápidas
    row = sdf.agg(
        F.count("*").alias("rows"),
        F.round(F.min(f"close_{exchange}"), 2).alias("price_min"),
        F.round(F.max(f"close_{exchange}"), 2).alias("price_max"),
        F.round(F.mean(f"volume_{exchange}"), 4).alias("vol_mean"),
    ).first()
    print(
        f"  Bronze ✓ {exchange:10s}  rows={row.rows:,}  "
        f"close=[{row.price_min:,.0f} – {row.price_max:,.0f}]  "
        f"vol_mean={row.vol_mean:.4f}"
    )
    return sdf


print("=== Leitura dos ficheiros Bronze ===")
sdf_binance  = read_and_rename("binance",  BRONZE_FILES["binance"])
sdf_kucoin    = read_and_rename("kucoin",    BRONZE_FILES["kucoin"])
sdf_coinbase = read_and_rename("coinbase", BRONZE_FILES["coinbase"])

print("\nSchema (Binance — idêntico para todos os exchanges):")
sdf_binance.printSchema()


=== Leitura dos ficheiros Bronze ===
  Bronze ✓ binance     rows=105,000  close=[15,650 – 68,719]  vol_mean=1100.6345
  Bronze ✓ kucoin      rows=105,895  close=[15,649 – 68,742]  vol_mean=78.6499
  Bronze ✓ coinbase    rows=105,278  close=[15,633 – 68,733]  vol_mean=192.4201

Schema (Binance — idêntico para todos os exchanges):
root
 |-- timestamp: timestamp (nullable = true)
 |-- open_binance: double (nullable = true)
 |-- high_binance: double (nullable = true)
 |-- low_binance: double (nullable = true)
 |-- close_binance: double (nullable = true)
 |-- volume_binance: double (nullable = true)



In [89]:
# ── Join temporal (inner) nas três exchanges ──────────────────────────────────
# O join inner em timestamp garante que só os instantes presentes nas três
# plataformas em simultâneo são retidos.
#
# NOTA DISTRIBUÍDA:
# Com 100k+ linhas, um join de três vias dispara um shuffle. Para evitar o
# sort-merge join e usar broadcast join (mais rápido para DataFrames menores),
# descomentar:
#   .join(F.broadcast(sdf_kucoin), on="timestamp", how="inner")
# (requer memória suficiente nos executores para o broadcast)

aligned = (
    sdf_binance
    .join(sdf_kucoin,    on="timestamp", how="inner")
    .join(sdf_coinbase, on="timestamp", how="inner")
)

# Remover linhas com NULLs residuais (páginas Parquet corruptas, etc.)
aligned = aligned.na.drop()

print(f"Linhas após alinhamento temporal : {aligned.count():,}")
print(f"Colunas                          : {len(aligned.columns)}")
aligned.show(5, truncate=True)


Linhas após alinhamento temporal : 104,978
Colunas                          : 16
+-------------------+------------+------------+-----------+-------------+--------------+-----------+-----------+----------+------------+-------------+-------------+-------------+------------+--------------+---------------+
|          timestamp|open_binance|high_binance|low_binance|close_binance|volume_binance|open_kucoin|high_kucoin|low_kucoin|close_kucoin|volume_kucoin|open_coinbase|high_coinbase|low_coinbase|close_coinbase|volume_coinbase|
+-------------------+------------+------------+-----------+-------------+--------------+-----------+-----------+----------+------------+-------------+-------------+-------------+------------+--------------+---------------+
|2021-01-01 00:00:00|    28923.63|     29017.5|   28690.17|      28752.8|    840.077569|    28924.9|    29015.0|   28687.1|     28750.9|  38.46912144|     28990.08|     29083.75|    28751.82|      28815.64|   319.29466145|
|2021-01-01 00:15:00|     2

---
### 2.3 Feature Engineering com Window Functions

Todo o feature engineering corre *dentro do Spark* — sem Pandas, sem `.collect()`.

| Feature | Descrição |
|---|---|
| `returns_<exch>` | Retorno aritmético bar-a-bar (close/close anterior − 1) |
| `sma7_<exch>` | Média móvel simples 7 barras |
| `sma30_<exch>` | Média móvel simples 30 barras |
| `volatility7_<exch>` | Desvio-padrão dos retornos em 7 barras (volatilidade realizada) |
| `vol_sma7_<exch>` | Média móvel do volume em 7 barras |
| `vol_ratio_<exch>` | Volume corrente / média móvel volume (ratio de atividade) |
| `price_range_<exch>` | (high − low) / open — range intra-barra normalizado |
| `gap_*` | Divergência de preço entre cada par de exchanges |
| `max_cross_gap` | Máximo absoluto dos gaps inter-exchange |


In [90]:
# ── Especificações de janela ──────────────────────────────────────────────────
# W_ORD  — ordenação sem partição (único símbolo BTC/USDT no DataFrame)
# W_7    — janela deslizante de 7 barras (barra actual + 6 anteriores)
# W_30   — janela deslizante de 30 barras
#
# NOTA: Num DataFrame multi-símbolo, adicionar .partitionBy("symbol") para
# impedir que as estatísticas de um símbolo contaminem outro.

W_ORD = Window.orderBy("timestamp")
W_7   = W_ORD.rowsBetween(-6,  0)
W_30  = W_ORD.rowsBetween(-29, 0)


def enrich_exchange(df, exch: str):
    """
    Calcula features por exchange usando PySpark Window functions.
    Lógica idêntica ao notebook original (1-para-1).

    Parâmetros
    ----------
    df   : pyspark.sql.DataFrame  DataFrame alinhado.
    exch : str                    Sufixo do exchange.
    """
    c = f"close_{exch}"
    v = f"volume_{exch}"

    # Retorno aritmético: (close_t − close_{t−1}) / close_{t−1}
    df = df.withColumn(
        f"returns_{exch}",
        (F.col(c) - F.lag(c, 1).over(W_ORD)) / F.lag(c, 1).over(W_ORD)
    )

    # SMAs (7 e 30 barras)
    df = df.withColumn(f"sma7_{exch}",  F.avg(c).over(W_7))
    df = df.withColumn(f"sma30_{exch}", F.avg(c).over(W_30))

    # Volatilidade realizada: std dos retornos em 7 barras (fórmula Bessel)
    df = df.withColumn(
        f"volatility7_{exch}",
        F.stddev(f"returns_{exch}").over(W_7)
    )

    # Features de volume
    df = df.withColumn(f"vol_sma7_{exch}", F.avg(v).over(W_7))
    df = df.withColumn(
        f"vol_ratio_{exch}",
        F.col(v) / F.when(
            F.col(f"vol_sma7_{exch}") == 0, None
        ).otherwise(F.col(f"vol_sma7_{exch}"))
    )

    # Range intra-barra normalizado pelo open
    df = df.withColumn(
        f"price_range_{exch}",
        (F.col(f"high_{exch}") - F.col(f"low_{exch}")) / F.col(f"open_{exch}")
    )
    return df


# Aplicar feature engineering nos três exchanges
features = aligned
for exch in ["binance", "kucoin", "coinbase"]:
    features = enrich_exchange(features, exch)
    print(f"  Features calculadas: {exch}")

# ── Features de divergência inter-exchange ────────────────────────────────────
# gap_A_B = (close_A − close_B) / close_A   (sinalizado, em termos fraccionários)
# max_cross_gap = max(|gap|) entre todos os pares

features = (
    features
    .withColumn(
        "gap_binance_kucoin",
        (F.col("close_binance") - F.col("close_kucoin")) / F.col("close_binance")
    )
    .withColumn(
        "gap_binance_coinbase",
        (F.col("close_binance") - F.col("close_coinbase")) / F.col("close_binance")
    )
    .withColumn(
        "gap_kucoin_coinbase",
        (F.col("close_kucoin") - F.col("close_coinbase")) / F.col("close_kucoin")
    )
    .withColumn(
        "max_cross_gap",
        F.greatest(
            F.abs("gap_binance_kucoin"),
            F.abs("gap_binance_coinbase"),
            F.abs("gap_kucoin_coinbase"),
        )
    )
)

# Remover linhas com NULLs das janelas iniciais (warm-up das rolling windows)
features = features.na.drop()

print(f"\n  DataFrame Silver: {features.count():,} linhas × {len(features.columns)} colunas")


  Features calculadas: binance
  Features calculadas: kucoin
  Features calculadas: coinbase

  DataFrame Silver: 104,953 linhas × 41 colunas


In [91]:
# ── Cache do DataFrame Silver ─────────────────────────────────────────────────
# .cache() evita que a Gold layer releia o Parquet e re-execute todas as
# Window functions quando o DataFrame 'features' é acedido múltiplas vezes.
# Em datasets que excedam a memória dos executores, substituir por .checkpoint()
# para spill para HDFS/S3 e quebrar a lineage.

features.cache()

# ── Escrita da camada Silver ──────────────────────────────────────────────────
silver_path = os.path.join(SILVER, "btc_aligned.parquet")
features.write.mode("overwrite").parquet(silver_path)
print(f"Camada Silver escrita → {silver_path}")


Camada Silver escrita → /home/jovyan/work/data/silver/btc_aligned.parquet


In [92]:
print("=== 1. CONTAGEM ANTES DO JOIN ===")
print("Linhas Binance: ", sdf_binance.count())
print("Linhas kucoin:   ", sdf_kucoin.count())
print("Linhas Coinbase:", sdf_coinbase.count())

print("\n=== 2. AMOSTRA DO TEMPO (Primeiras 3 linhas) ===")
print("Binance:")
sdf_binance.select("timestamp").show(3)
print("kucoin:")
sdf_kucoin.select("timestamp").show(3)
print("Coinbase:")
sdf_coinbase.select("timestamp").show(3)

print("\n=== 3. CONTAGEM DEPOIS DO JOIN ===")
print("Linhas Alinhadas:", aligned.count())

=== 1. CONTAGEM ANTES DO JOIN ===
Linhas Binance:  105000
Linhas kucoin:    105895
Linhas Coinbase: 105278

=== 2. AMOSTRA DO TEMPO (Primeiras 3 linhas) ===
Binance:
+-------------------+
|          timestamp|
+-------------------+
|2021-01-01 00:00:00|
|2021-01-01 00:15:00|
|2021-01-01 00:30:00|
+-------------------+
only showing top 3 rows

kucoin:
+-------------------+
|          timestamp|
+-------------------+
|2021-01-01 00:00:00|
|2021-01-01 00:15:00|
|2021-01-01 00:30:00|
+-------------------+
only showing top 3 rows

Coinbase:
+-------------------+
|          timestamp|
+-------------------+
|2021-01-01 00:00:00|
|2021-01-01 00:15:00|
|2021-01-01 00:30:00|
+-------------------+
only showing top 3 rows


=== 3. CONTAGEM DEPOIS DO JOIN ===
Linhas Alinhadas: 104978


---
## 3. Resultados — Deteção de Anomalias

### 3.1 Método A — Z-score Distribuído (Spark SQL + Window)

Para cada feature *f*, o Z-score global é:

$$z_f = \left| \frac{f - \mu_f}{\sigma_f} \right|$$

onde μ_f e σ_f são calculados numa **única passagem de agregação** sobre o DataFrame completo.
Os escalares resultantes são embebidos nas expressões de coluna via `F.lit()`,
que o Spark transmite automaticamente a todos os executores em tempo de compilação do plano —
sem necessidade de `sc.broadcast()` explícito para valores escalares.

Uma linha é sinalizada quando o maior |z| entre todas as features excede `Z_THRESH`.


In [93]:
# ── Agregação global (uma única passagem Spark) ───────────────────────────────
stats_row = features.select(
    *[F.mean(c).alias(f"mean_{c}")  for c in Z_FEATURES],
    *[F.stddev(c).alias(f"std_{c}") for c in Z_FEATURES],
).first()

print("Estatísticas globais para normalização Z-score:")
for col in Z_FEATURES:
    mu  = stats_row[f"mean_{col}"]
    sig = stats_row[f"std_{col}"]
    print(f"  {col:30s}  μ={mu: .8f}  σ={sig:.8f}")

# ── Adicionar colunas Z-score via F.lit() (broadcast automático) ──────────────
scored_z = features
z_abs_cols = []
for col in Z_FEATURES:
    mu  = float(stats_row[f"mean_{col}"])
    sig = float(stats_row[f"std_{col}"])
    sig = max(sig, 1e-10)   # protecção contra features com variância zero
    z_name = f"z_{col}"
    scored_z = scored_z.withColumn(
        z_name,
        F.abs((F.col(col) - F.lit(mu)) / F.lit(sig))
    )
    z_abs_cols.append(z_name)

# Z-score máximo absoluto entre todas as features (nível de linha)
scored_z = scored_z.withColumn("zscore_max", F.greatest(*z_abs_cols))

# Flag binária: 1 se alguma feature exceder o limiar
scored_z = scored_z.withColumn(
    "anomaly_zscore",
    F.when(F.col("zscore_max") > Z_THRESH, 1)
     .otherwise(0)
     .cast(IntegerType())
)

n_z = scored_z.filter(F.col("anomaly_zscore") == 1).count()
print(f"\nAnomalias Z-score (limiar |z|>{Z_THRESH}): {n_z}")

scored_z.filter(F.col("anomaly_zscore") == 1) \
        .select("timestamp", "close_binance", "zscore_max",
                *z_abs_cols[:3], "anomaly_zscore") \
        .orderBy("timestamp") \
        .show(20, truncate=False)


Estatísticas globais para normalização Z-score:
  returns_binance                 μ= 0.00001051  σ=0.00371289
  volatility7_binance             μ= 0.00279147  σ=0.00247802
  vol_ratio_binance               μ= 1.01276726  σ=0.47819716
  max_cross_gap                   μ= 0.00060934  σ=0.00087610
  price_range_binance             μ= 0.00464823  σ=0.00451560

Anomalias Z-score (limiar |z|>3.5): 4032
+-------------------+-------------+------------------+--------------------+---------------------+-------------------+--------------+
|timestamp          |close_binance|zscore_max        |z_returns_binance   |z_volatility7_binance|z_vol_ratio_binance|anomaly_zscore|
+-------------------+-------------+------------------+--------------------+---------------------+-------------------+--------------+
|2021-01-01 01:00:00|29382.59     |3.5962399916572987|3.5962399916572987  |1.0110475465646929   |2.366153329679832  |1             |
|2021-01-02 00:15:00|29032.24     |4.090518346721295 |2.550914052730

---
### 3.2 Método B — Pipeline ML Distribuído: BisectingKMeans

**Porquê BisectingKMeans em vez de Isolation Forest?**

O notebook original usava `sklearn.IsolationForest` via `.toPandas()`, o que:
- (a) Transfere **todo o dataset para o driver** — risco de OOM com 100k+ linhas.
- (b) Corre **single-threaded** num único nó — desperdiça o cluster.
- (c) **Quebra a lineage** do Spark, impedindo optimizações AQE.

`pyspark.ml.clustering.BisectingKMeans` corre inteiramente dentro do Spark:
- O treino itera em paralelo em todos os executores.
- O `.transform()` (predição) é um *distributed map* — sem `.collect()`.
- O resultado é uma coluna Spark, não uma Series Pandas.

**Estratégia de identificação de anomalias:**

Após o fit, `clusterSizes()` devolve a população de cada um dos K clusters folha.
Barras anómalas (flash crashes, volume spikes) tendem a formar clusters muito pequenos
e isolados longe do centróide da massa de dados. Qualquer cluster com dimensão relativa
≤ `ANOMALY_CLUSTER_FRAC` (2%) é classificado como anómalo — espelhando `contamination=0.02`
do Isolation Forest original.

**Pipeline ML:**
```
VectorAssembler  →  StandardScaler  →  BisectingKMeans
  (transformer)       (estimator)         (estimator)
```


In [94]:
# ── Etapa 1: VectorAssembler ──────────────────────────────────────────────────
# Combina as colunas de features num único DenseVector ("features_raw").
# handleInvalid="skip" remove linhas com NULLs residuais em qualquer feature.

assembler = VectorAssembler(
    inputCols=ML_FEATURES,
    outputCol="features_raw",
    handleInvalid="skip",
)

# ── Etapa 2: StandardScaler (distribuído) ────────────────────────────────────
# Calcula μ e σ numa passagem distribuída (fit) e aplica a escala via map
# distribuído (transform). Equivalente ao sklearn.StandardScaler.

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features_scaled",
    withMean=True,
    withStd=True,
)

# ── Etapa 3: BisectingKMeans ─────────────────────────────────────────────────
# Algoritmo hierárquico top-down: começa com um cluster e bissecta iterativamente
# o cluster com maior SSE até atingir K clusters folha.
# Parâmetros:
#   k        — número de clusters folha alvo
#   maxIter  — iterações máximas de bissecção por passo
#   seed     — reprodutibilidade

bkm = BisectingKMeans(
    featuresCol="features_scaled",
    predictionCol="cluster_id",
    k=BKM_K,
    maxIter=BKM_MAX_ITER,
    seed=BKM_SEED,
    distanceMeasure="euclidean",
)

# ── Pipeline e treino ─────────────────────────────────────────────────────────
# Pipeline.fit() treina todos os estimadores em sequência.
# Tudo corre dentro do grafo de execução Spark — sem dados no driver.

pipeline = Pipeline(stages=[assembler, scaler, bkm])

print("A treinar pipeline ML (VectorAssembler → StandardScaler → BisectingKMeans)...")
pipeline_model = pipeline.fit(scored_z)
print("Pipeline treinado com sucesso.")


A treinar pipeline ML (VectorAssembler → StandardScaler → BisectingKMeans)...
Pipeline treinado com sucesso.


In [95]:
# ── Identificação dos clusters anómalos ──────────────────────────────────────
# .transform() é uma operação distribuída lazy — não move dados para o driver.
# Adiciona as colunas: features_raw, features_scaled, cluster_id

scored_km = pipeline_model.transform(scored_z)

# Em vez de dependermos do bkm_model.clusterSizes(), vamos contar na perfeição 
# usando o motor distribuído do Spark:
tamanhos_df = scored_km.groupBy("cluster_id").count().orderBy("cluster_id").collect()

# clusterSizes() devolve uma lista Python: índice = cluster_id, valor = nº linhas
cluster_sizes = {row["cluster_id"]: row["count"] for row in tamanhos_df}
total_rows    = sum(cluster_sizes.values())

print(f"Total de linhas avaliadas : {total_rows:,}")
print("\nDimensão dos clusters (cluster_id → linhas → dimensão relativa):")

anomaly_cluster_ids = set()
for cid, sz in cluster_sizes.items():
    rel  = sz / total_rows
    flag = " ← CLUSTER ANÓMALO" if rel <= ANOMALY_CLUSTER_FRAC else ""
    print(f"  cluster {cid:2d} : {sz:8,} linhas  ({rel:.4%}){flag}")
    if rel <= ANOMALY_CLUSTER_FRAC:
        anomaly_cluster_ids.add(cid)

print(f"\nIDs de clusters anómalos (dimensão ≤ {ANOMALY_CLUSTER_FRAC:.0%}): {anomaly_cluster_ids}")


Total de linhas avaliadas : 104,953

Dimensão dos clusters (cluster_id → linhas → dimensão relativa):
  cluster  0 :   34,987 linhas  (33.3359%)
  cluster  1 :   38,709 linhas  (36.8822%)
  cluster  2 :   12,592 linhas  (11.9978%)
  cluster  3 :    2,175 linhas  (2.0724%)
  cluster  4 :    1,477 linhas  (1.4073%) ← CLUSTER ANÓMALO
  cluster  5 :    7,457 linhas  (7.1051%)
  cluster  6 :    6,628 linhas  (6.3152%)
  cluster  7 :      928 linhas  (0.8842%) ← CLUSTER ANÓMALO

IDs de clusters anómalos (dimensão ≤ 2%): {4, 7}


In [96]:
# ── Flag anomaly_kmeans via array literal broadcast ───────────────────────────
# F.array(*[F.lit(cid) for cid in ...]) cria um literal ArrayType avaliado
# localmente em cada executor — sem comunicação com o driver por linha.
# F.array_contains() testa pertença ao array de IDs anómalos.

if anomaly_cluster_ids:
    anomaly_ids_lit = F.array(*[F.lit(int(cid)) for cid in anomaly_cluster_ids])

    scored_km = scored_km.withColumn(
        "anomaly_kmeans",
        F.when(
            F.array_contains(anomaly_ids_lit, F.col("cluster_id")), 1
        ).otherwise(0).cast(IntegerType())
    )
else:
    print("AVISO: Nenhum cluster atingiu o limiar de anomalia. "
          "Considere aumentar BKM_K ou diminuir ANOMALY_CLUSTER_FRAC.")
    scored_km = scored_km.withColumn(
        "anomaly_kmeans",
        F.lit(0).cast(IntegerType())
    )

n_km = scored_km.filter(F.col("anomaly_kmeans") == 1).count()
print(f"Anomalias BisectingKMeans sinalizadas: {n_km}")

scored_km.filter(F.col("anomaly_kmeans") == 1) \
         .select("timestamp", "close_binance", "cluster_id",
                 "anomaly_kmeans", "zscore_max") \
         .orderBy("timestamp") \
         .show(20, truncate=False)


Anomalias BisectingKMeans sinalizadas: 2405
+-------------------+-------------+----------+--------------+------------------+
|timestamp          |close_binance|cluster_id|anomaly_kmeans|zscore_max        |
+-------------------+-------------+----------+--------------+------------------+
|2021-01-01 01:00:00|29382.59     |7         |1             |3.5962399916572987|
|2021-01-02 00:15:00|29032.24     |4         |1             |4.090518346721295 |
|2021-01-02 12:15:00|30289.7      |7         |1             |7.7769659606596875|
|2021-01-02 12:30:00|30695.06     |7         |1             |3.601581759623967 |
|2021-01-02 13:30:00|31278.54     |7         |1             |3.7802202689218545|
|2021-01-02 15:30:00|31640.65     |7         |1             |3.325340155913036 |
|2021-01-02 16:00:00|32029.09     |7         |1             |3.7334468809994523|
|2021-01-02 16:30:00|32583.98     |7         |1             |3.8361252455748813|
|2021-01-02 20:00:00|32785.24     |4         |1             |4.07

---
### 3.3 Flag Combinada `anomaly_any`

`anomaly_any = 1` quando **qualquer um** dos dois detectores sinaliza a linha.
Esta abordagem OR maximiza o recall: se um método falha um evento, o outro pode captá-lo.


In [97]:
# ── Flag combinada ────────────────────────────────────────────────────────────
final = scored_km.withColumn(
    "anomaly_any",
    F.when(
        (F.col("anomaly_zscore") == 1) | (F.col("anomaly_kmeans") == 1), 1
    ).otherwise(0).cast(IntegerType())
)

n_any = final.filter(F.col("anomaly_any") == 1).count()

print("=" * 50)
print("RESUMO DE FLAGS DE ANOMALIA")
print("=" * 50)
print(f"  Z-score (|z| > {Z_THRESH})      : {n_z}")
print(f"  BisectingKMeans              : {n_km}")
print(f"  Combinada (any)              : {n_any}")


RESUMO DE FLAGS DE ANOMALIA
  Z-score (|z| > 3.5)      : 4032
  BisectingKMeans              : 2405
  Combinada (any)              : 4715


---
### 3.4 Escrita da Camada Gold

O DataFrame Gold é particionado por `anomaly_any`:
- `anomaly_any=0/` — barras normais
- `anomaly_any=1/` — barras sinalizadas

Esta estratégia permite que pipelines downstream (dashboards, jobs de alerta)
façam *push-down* do predicado e leiam apenas a partição `anomaly_any=1`
em vez de varrer todo o dataset.

As colunas vetoriais intermediárias (`features_raw`, `features_scaled`) são
removidas antes da escrita para evitar inflar o Parquet com blobs binários
sem valor analítico.


In [98]:
# Remover colunas vectoriais intermédias (sem valor analítico no Gold)
gold_df = final.drop("features_raw", "features_scaled")

gold_path = os.path.join(GOLD, "btc_anomaly_scored.parquet")
(
    gold_df
    .write
    .mode("overwrite")
    .partitionBy("anomaly_any")   # predicado push-down para scans de anomalias
    .parquet(gold_path)
)
print(f"Camada Gold escrita → {gold_path}")

# ── Validação rápida por leitura ──────────────────────────────────────────────
print("\nVerificação (Gold):")
gold = spark.read.parquet(gold_path)
gold.select(
    "timestamp", "close_binance", "zscore_max",
    "cluster_id", "anomaly_zscore", "anomaly_kmeans", "anomaly_any"
).orderBy("timestamp").show(10, truncate=False)

print("\nSchema Gold:")
gold.printSchema()


Camada Gold escrita → /home/jovyan/work/data/gold/btc_anomaly_scored.parquet

Verificação (Gold):
+-------------------+-------------+------------------+----------+--------------+--------------+-----------+
|timestamp          |close_binance|zscore_max        |cluster_id|anomaly_zscore|anomaly_kmeans|anomaly_any|
+-------------------+-------------+------------------+----------+--------------+--------------+-----------+
|2021-01-01 00:30:00|28930.11     |2.9583992715817993|2         |0             |0             |0          |
|2021-01-01 00:45:00|28995.13     |2.284127151777112 |2         |0             |0             |0          |
|2021-01-01 01:00:00|29382.59     |3.5962399916572987|7         |1             |1             |1          |
|2021-01-01 01:15:00|29385.39     |1.815298751656877 |6         |0             |0             |0          |
|2021-01-01 01:30:00|29319.87     |2.700335517207437 |0         |0             |0             |0          |
|2021-01-01 01:45:00|29409.99     |2.4

---
### 3.5 Estatísticas Descritivas dos Scores


In [99]:
print("Estatísticas descritivas dos scores de anomalia:")
gold.select(
    F.round(F.mean("zscore_max"),    4).alias("z_mean"),
    F.round(F.stddev("zscore_max"),  4).alias("z_std"),
    F.round(F.max("zscore_max"),     4).alias("z_max"),
    F.round(F.mean("cluster_id"),    4).alias("cluster_mean"),
    F.round(F.stddev("cluster_id"),  4).alias("cluster_std"),
).show(truncate=False)

print("\nDistribuição de anomalias por flag:")
gold.groupBy("anomaly_zscore", "anomaly_kmeans", "anomaly_any") \
    .count() \
    .orderBy("anomaly_any", "anomaly_zscore", "anomaly_kmeans") \
    .show()


Estatísticas descritivas dos scores de anomalia:
+------+------+-------+------------+-----------+
|z_mean|z_std |z_max  |cluster_mean|cluster_std|
+------+------+-------+------------+-----------+
|1.2157|1.2427|56.9519|1.5233      |1.8501     |
+------+------+-------+------------+-----------+


Distribuição de anomalias por flag:
+--------------+--------------+-----------+------+
|anomaly_zscore|anomaly_kmeans|anomaly_any| count|
+--------------+--------------+-----------+------+
|             0|             0|          0|100238|
|             0|             1|          1|   683|
|             1|             0|          1|  2310|
|             1|             1|          1|  1722|
+--------------+--------------+-----------+------+



---
## 4. Conclusões

### 4.1 Interpretação dos Resultados

O pipeline combina dois métodos complementares totalmente distribuídos:

**Z-score distribuído** (Spark SQL) deteta desvios extremos em features individuais.
É interpretável e computacionalmente barato, mas assume que as distribuições são
aproximadamente normais — pressuposto violado em *flash crashes* (caudas muito pesadas).

**BisectingKMeans** (pyspark.ml) não assume nenhuma distribuição e é robusto a outliers
multivariados. Barras anómalas formam clusters pequenos e isolados no espaço de features
normalizado. A identificação de clusters anómalos por dimensão relativa espelha
o parâmetro `contamination` do Isolation Forest sem requerer `.toPandas()`.

A **combinação das duas flags** (`anomaly_any`) maximiza o recall.

### 4.2 Garantias de Processamento Distribuído

| Operação | Abordagem distribuída |
|---|---|
| Leitura Bronze | `spark.read.parquet()` — paralelo por ficheiro/partição |
| Join temporal | Sort-merge join (ou broadcast com hint) — distribuído |
| Feature engineering | PySpark Window functions — paralelo por partição |
| Z-score stats | Única passagem de agregação Spark — sem collect |
| Z-score scoring | `F.lit()` broadcast automático — map distribuído |
| StandardScaler | pyspark.ml fit + transform — distribuído |
| BisectingKMeans | pyspark.ml fit + transform — distribuído |
| Flag anomaly_kmeans | `F.array_contains()` — avaliação local em cada executor |
| Escrita Gold | `write.parquet()` particionado — paralelo por partição |
| **`.toPandas()` no caminho de scoring** | **ZERO invocações** |

### 4.3 Limitações

- O BisectingKMeans identifica anomalias por dimensão de cluster, não por distância ao centróide.
  Para uma abordagem baseada em distância, calcular `clusterCenters()` e medir a distância
  euclidiana de cada ponto ao centróide do seu cluster (fully distributed com Window functions).
- O pipeline é batch — não deteta anomalias em tempo real.
  Para streaming, usar **Spark Structured Streaming** com Z-score incremental em janelas deslizantes.
- `BKM_K` e `ANOMALY_CLUSTER_FRAC` são hiper-parâmetros que devem ser calibrados por validação.

### 4.4 Trabalho Futuro

- Integrar Structured Streaming para deteção em tempo real (janelas VWAP de 1h).
- Usar **MLflow** para registar `Z_THRESH`, `BKM_K`, `ANOMALY_CLUSTER_FRAC` e métricas por execução.
- Substituir a identificação por dimensão por distância ao centróide para um score contínuo de anomalia.
- Adicionar particionamento por `year`/`month` no Gold para acesso temporal eficiente.


In [100]:
# ── Resumo final do pipeline ──────────────────────────────────────────────────
print("=" * 65)
print("RESUMO DO PIPELINE — Tema 5: Anomalias Multi-Exchange (Big Data)")
print("=" * 65)
print(f"  Exchanges          : Binance, kucoin, Coinbase")
print(f"  Linhas alinhadas   : {features.count():,}")
print(f"  Features ML        : {len(ML_FEATURES)}")
print(f"  Flags Z-score      : {n_z}  (|z| > {Z_THRESH})")
print(f"  Clusters KMeans    : {BKM_K}  (anómalos ≤ {ANOMALY_CLUSTER_FRAC:.0%})")
print(f"  Flags KMeans       : {n_km}")
print(f"  Flags combinadas   : {n_any}")
print()
print("Outputs:")
print(f"  Silver → {silver_path}")
print(f"  Gold   → {gold_path}")
print()
print("Processamento distribuído:")
print("  • ZERO invocações de .toPandas() no caminho de scoring")
print("  • Z-score: única passagem de agregação + broadcast F.lit()")
print("  • BisectingKMeans: fit + transform totalmente dentro do Spark")
print("  • Gold Parquet particionado por anomaly_any")

# ── Libertar cache e terminar sessão ─────────────────────────────────────────
features.unpersist()
spark.stop()
print("\nSparkSession terminada. Pipeline concluído.")


RESUMO DO PIPELINE — Tema 5: Anomalias Multi-Exchange (Big Data)
  Exchanges          : Binance, kucoin, Coinbase
  Linhas alinhadas   : 104,953
  Features ML        : 9
  Flags Z-score      : 4032  (|z| > 3.5)
  Clusters KMeans    : 8  (anómalos ≤ 2%)
  Flags KMeans       : 2405
  Flags combinadas   : 4715

Outputs:
  Silver → /home/jovyan/work/data/silver/btc_aligned.parquet
  Gold   → /home/jovyan/work/data/gold/btc_anomaly_scored.parquet

Processamento distribuído:
  • ZERO invocações de .toPandas() no caminho de scoring
  • Z-score: única passagem de agregação + broadcast F.lit()
  • BisectingKMeans: fit + transform totalmente dentro do Spark
  • Gold Parquet particionado por anomaly_any

SparkSession terminada. Pipeline concluído.
